In [ ]:
# 第一步：为新增的两路生成答案，以 append 追加到目标表，得分为空。
import asyncio
import html
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import HTML, Image, display

# 1. 环境与输入。执行本格会调用模型并写入本次运行的表。
PROJECT = Path('/yzp/zhaozy/yangzepeng/0905/demiwtg')
sys.path.insert(0, str(PROJECT)) if str(PROJECT) not in sys.path else None
DATA_ROOT = PROJECT.parent
os.environ['DEMIWTG_DATASETS_ROOT'] = str(DATA_ROOT)

from demiflow import data
from demiflow.lance.blobs import BlobRef
from evaluation.t2i.v2.t2i_v2_eval_pipeline import DATASETS, config, run_pipeline

# 第一步独立配置：从 INPUT_TABLE_URI 的固定版本读题，写入 OUTPUT_TABLE_URI。
INPUT_TABLE_URI = 'demiwtg/benchmark/t2i/v2/datasets/candidates__glm53_authoring_20260925_04.lance'
INPUT_VERSION = 1
OUTPUT_TABLE_URI = 'demiwtg/evaluation/t2i/v2/datasets/results__three_models_answers_20260925_02.lance'
WRITE_MODE = 'append'  # 'append' 追加本次结果；'overwrite' 覆盖目标表。
RUN_ID = 'qwen21_bagel_references_20260925_01'  # 更换题表或答题模型参数后使用新运行名。

# 2. 本次只运行新增的两路：参考知识与参考图均来自输入题表的 references_json。
# 本地模型依次使用 GPU1；各自的依赖只在对应子进程生效。
ANSWER_MODELS = [
    {'backend': 'diffusers', 'model': 'Qwen-Image-2.1', 'use_references': True,
     'model_path': str(DATA_ROOT / 'models/Qwen-Image-2.1'),
     'python': str(DATA_ROOT / 'env/bin/python'), 'cuda_visible_devices': '1',
     # 沿用 Edit 流程已验证的官方 Diffusers 源码及专用依赖。
     'pythonpath': ['/tmp/edit_qwen21_dependencies', '/tmp/edit_diffusers_official/src'],
     'device': 'cuda:0', 'seed': 42,
     'parameters': {'num_inference_steps': 40, 'output_resolution': 1024}},
    {'backend': 'bagel', 'model': 'BAGEL-7B-MoT', 'use_references': True,
     'model_path': str(DATA_ROOT / 'models/BAGEL-7B-MoT'),
     'python': str(DATA_ROOT / 'env-bagel/bin/python'), 'cuda_visible_devices': '1',
     'seed': 42, 'parameters': {}},
]

CONFIG = config(answer_models=ANSWER_MODELS)

# 此格只调用答题模型；生成后返回目标表的实际路径和固定版本。
state = await asyncio.to_thread(run_pipeline, DATA_ROOT / DATASETS / RUN_ID,
                                {'uri': INPUT_TABLE_URI, 'version': INPUT_VERSION}, CONFIG,
                                target_uri=OUTPUT_TABLE_URI, write_mode=WRITE_MODE)
display(state)
OUTPUT_VERSION = state['target']['version']
print('输出表：', OUTPUT_TABLE_URI, '版本：', OUTPUT_VERSION)
results = data.read_lance(str(DATA_ROOT / OUTPUT_TABLE_URI), version=OUTPUT_VERSION).take_all()
for row in results:
    display(HTML('<p>' + html.escape(row['instruction']) + ' — ' + html.escape(row['answer_model']) + '</p>'))
    if row['image_json']:
        display(Image(data=BlobRef(**json.loads(row['image_json'])).read(DATA_ROOT), width=512))
    display({'status': row['status'], 'reason': row['reason']})


In [ ]:
# 第二步：读取已有答案，judge 后写入指定目标表。可独立运行。
import asyncio
import html
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import HTML, Image, display

# 环境：本格只调用 judge，不加载答题模型。
PROJECT = Path('/yzp/zhaozy/yangzepeng/0905/demiwtg')
sys.path.insert(0, str(PROJECT)) if str(PROJECT) not in sys.path else None
DATA_ROOT = PROJECT.parent
os.environ['DEMIWTG_DATASETS_ROOT'] = str(DATA_ROOT)

from demiflow import data
from demiflow.lance.blobs import BlobRef
from evaluation.t2i.v2.t2i_v2_eval_pipeline import DATASETS, config, run_judging

# 独立配置源表、固定版本、目标表；需要覆盖源表时，将两个 URI 填为相同路径。

# 默认复用已有的 12 张答案；新答题时，填写第一格的输出路径与版本。
INPUT_TABLE_URI = 'demiwtg/evaluation/t2i/v2/datasets/answers__three_models_gpt6_sol_20260925_01.lance'
INPUT_VERSION = 13
OUTPUT_TABLE_URI = 'demiwtg/evaluation/t2i/v2/datasets/scores__gpt6_sol_judge_20260925_02.lance'
WRITE_MODE = 'overwrite'  # 'append' 追加本次结果；'overwrite' 覆盖目标表。
JUDGE_RUN_ID = 'gpt6_sol_judge_20260925_02'  # 修复接口后重试或更换 judge 时使用新名，避免复用旧失败响应。
JUDGE_MODEL = {'model': 'openrouter/openai/gpt-6-sol', 'base_url': 'http://127.0.0.1:4001/v1',
               'api_key_env': 'MODELHUB_API_KEY', 'max_output_tokens': 8192, 'timeout_s': 600}
CONFIG = config(judge_model=JUDGE_MODEL)

# 先读取和查看源表，再执行 judge；两步之间不依赖内存中的生成模型或答案。
source = {'uri': str(DATA_ROOT / INPUT_TABLE_URI), 'version': INPUT_VERSION}
results = data.read_lance(str(DATA_ROOT / INPUT_TABLE_URI), version=INPUT_VERSION).take_all()
display(HTML('<style>table.t2i-preview {width:100%;table-layout:fixed;} table.t2i-preview th, table.t2i-preview td {text-align:left;vertical-align:top;white-space:pre-wrap;overflow-wrap:anywhere;}</style>'))
display(HTML(pd.DataFrame(results, columns=['task_id', 'answer_model', 'instruction', 'status']).to_html(index=False, escape=True, classes='t2i-preview')))
state = await asyncio.to_thread(run_judging, DATA_ROOT / DATASETS / JUDGE_RUN_ID, source, CONFIG, target_uri=OUTPUT_TABLE_URI, write_mode=WRITE_MODE)
display(state)

# 读取目标表的固定版本，展示完整评分与理由；图片引用保持不变。
SCORED_VERSION = state['target']['version']
print('评分目标表：', OUTPUT_TABLE_URI, '新版本：', SCORED_VERSION)
results = data.read_lance(str(DATA_ROOT / OUTPUT_TABLE_URI), version=SCORED_VERSION).take_all()
for row in results:
    display(HTML('<p><b>' + html.escape(row['concept'] or '') + '</b>：' + html.escape(row['instruction']) + '</p>'))
    if row['image_json']:
        display(Image(data=BlobRef(**json.loads(row['image_json'])).read(DATA_ROOT), width=512))
    display(HTML(pd.DataFrame([{key: row[key] for key in ('answer_model', 'status', 'reason', 'alignment_score', 'quality_score', 'aesthetics_score')}]).to_html(index=False, escape=True, classes='t2i-preview')))
    if row['judge_json']:
        judgment = json.loads(row['judge_json'])
        display(HTML(pd.DataFrame([{'维度': dimension, '项目': key, '分数': value, '理由': judgment[dimension + '_reasons'][key]}
                                  for dimension in ('alignment', 'quality', 'aesthetics') for key, value in judgment[dimension].items()])
                     .to_html(index=False, escape=True, classes='t2i-preview')))
